# ocd-pomdp: example usage

A walkthrough of the core API: load a config, instantiate a POMDP, solve it, visualize its policy, simulate data from it, fit it to a real subject, simulate an ensemble from already-fitted parameters, and check recovery diagnostics.

Every cell below was validated against this repo's real data before being written here. See `README.md` for the cluster/job-array workflow this all plugs into.

In [ ]:
import sys, os
import numpy as np
import pandas as pd

# The notebook runs from notebooks/, so the repository root is one level up.
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

## 1. Configuration

`src/config` builds a `SimulationConfig` once per process from whichever override file `SIM_CONFIG_PATH` points to (or the bundled default if unset). Every constant the model code expects (`TAU`, `PARAM_RANGES`, `DATA_PATH`, ...) is exposed both as a bare name and as `CONFIG.<name>`.

Running this notebook with no `SIM_CONFIG_PATH` set gives you the default override (`src/config/simulation_params.py`).

In [ ]:
from src.config import CONFIG, PARAM_RANGES, PARAM_ORDER, ALGORITHM, POMDP_TYPE

print("TASK:", CONFIG.TASK)
print("ALGORITHM:", ALGORITHM, "| POMDP_TYPE:", POMDP_TYPE)
print("PARAM_RANGES:", PARAM_RANGES)
print("PARAM_ORDER:", PARAM_ORDER)

To use a specific sweep config instead of the default, load it explicitly with `load_config` rather than relying on the env var (handy inside a notebook, where you don't want to restart the kernel to switch configs):

In [ ]:
from src.config.loader import load_config

other_cfg = load_config(
    os.path.join(project_root, "data/simulation_configs/simulation_params_LBE-T-RPhCL--.py")
)
print("Loaded override TASK:", other_cfg.TASK, "| PARAM_ORDER:", other_cfg.PARAM_ORDER)


## 2. Instantiate a POMDP model

`POMDPFactory()` picks the model class based on `POMDP_TYPE` in the config (`"exaggerate"`, `"urgency"`, `"forgetting"`, ...) and constructs it with the config's default parameters.

In [ ]:
from src.pomdp import POMDPFactory

pomdp = POMDPFactory()
print(type(pomdp).__name__, "| horizon:", pomdp.horizon_condition, "| max_draws:", pomdp.max_draws)

You can also pass explicit parameters instead of the config defaults, exactly like `generate_meg_data` does internally (`pomdp.__init__(**params)`). **Different `POMDP_TYPE`s accept different parameters** — `vanilla` (`POMDP`) only takes `tau`, `xi`, `hazard_lapse`, `subjective_cost`, plus the structural args below; it does *not* accept `patience`, `urgency_coefficient`, `c_max`, `belief_bias`, `exaggeration_factor`, or `gamma` (those belong to `urgency`/`exaggerate`/`forgetting`). Passing a key the active type doesn't accept raises `TypeError` — `src/pomdp/validation.py` checks this automatically for PARAM_RANGES against whatever config is loaded.

In [ ]:
params = {
    "tau": 5.0,
    "xi": 0.05,
    "subjective_cost": -50,
    "horizon_condition": "short",
    "max_cards_per_draw": 5,
    "verbose": False,
}
pomdp = POMDPFactory()
pomdp.__init__(**params)
print(type(pomdp).__name__, "| horizon:", pomdp.horizon_condition, "| max_draws:", pomdp.max_draws)

To use a richer model with `patience`/`urgency_coefficient`/`belief_bias`/`exaggeration_factor`, select its `POMDP_TYPE` explicitly via `POMDPFactory("exaggerate")` rather than relying on the default — this is also why sweep configs in `data/simulation_configs/` set `POMDP_TYPE` explicitly rather than relying on `vanilla`:

In [ ]:
exaggerate_params = {
    **params,
    "patience": 6,
    "c_max": 20,
    "belief_bias": 1.0,
    "exaggeration_factor": 1.5,
}
exaggerate_pomdp = POMDPFactory("exaggerate")
exaggerate_pomdp.__init__(**exaggerate_params)
print(type(exaggerate_pomdp).__name__, "| horizon:", exaggerate_pomdp.horizon_condition)

## 3. Solve the model (value iteration)

Every parameter set needs its own solve before you can simulate or score it against data, `value_iteration()` fills in `best_actions`, the optimal policy at every (draw, belief) state.

In [ ]:
pomdp.value_iteration()
print("best_actions shape (draw, prev_yellow, curr_yellow):", pomdp.best_actions.shape)

## 4. Simulate a full multi-trial dataset from the model

`generate_meg_data` is the higher-level helper `fit_data.py`/`simulate_data` use under the hood: it instantiates a POMDP from `params`, solves it once, then simulates `num_trials` trials (or replays real `data` if given) and returns a tidy per-trial DataFrame.

In [ ]:
from src.params_fitting.data_simulation import generate_meg_data

sim_data, df_ev, best_actions = generate_meg_data(params, data=None, num_trials=10)
sim_data[["num_draws", "outcome"]].head()

## 5. Simulate an ensemble from already-fitted parameters

Once a config has been fit (see `PIPELINE.txt`), `results.pkl` holds each subject's fitted `fit_params_ga`, and `all_simulated_data.pkl` holds a full re-simulation of every subject under those parameters (`scripts/fit_data.py` builds it via `simulate_data`), this is what the GLM ensemble scripts (`glm_multiprocessed_*.py`) build on.

**Note:** `simulate_data`/`generate_meg_data` always build their POMDP via the bare `POMDPFactory()`, i.e. whatever `POMDP_TYPE` this *process* resolved to at import time (here: `vanilla`, from section 1), not the `POMDP_TYPE` of whatever config you `load_config()` afterward. So you can't correctly re-simulate an `urgency`/`exaggerate` config's results inline like this unless the kernel itself was started with `SIM_CONFIG_PATH` pointing at that config. We use `LB-XT-RPHCLUK` (`urgency`) below purely to read its *already-computed* `all_simulated_data.pkl`, not to call `simulate_data` fresh.

In [ ]:
# These cells need a completed fit under data/POMDP/, which a fresh clone does
# not have. Run the fitting stage first, or read the deposited parameters in
# results/fits/ instead. See the README.
import os

fitted_cfg = load_config(os.path.join(
    project_root, 'data/simulation_configs/simulation_params_SB-XT-RPh----.py'))
HAVE_FIT = os.path.exists(fitted_cfg.RESULTS_PATH)
print('fitted results available:', HAVE_FIT)
if HAVE_FIT:
    results_df = pd.read_pickle(fitted_cfg.RESULTS_PATH)
    print('fitted subjects:', len(results_df))


## 6. Recovery diagnostics

`scripts/fit_data.py` doesn't just fit subjects, it also re-simulates data from the fitted parameters and re-fits *that* ("recovery"), so you can check whether the optimizer reliably recovers parameters it itself generated. `results_recovered.pkl` holds that second fit. `src/utils` has dedicated plotting helpers for this, same as `recovery_post_analysis.ipynb` uses; both take `is_latex=False` explicitly since (as above) there's no LaTeX available here.

In [47]:
if HAVE_FIT:
    from src.utils import plot_true_vs_recovered_params
    
    results_df_recovered = pd.read_pickle(fitted_cfg.RESULTS_RECOVERED_PATH)
    
    figure_path = os.path.join(project_root, f'figures/{fitted_cfg.TASK}')
    os.makedirs(figure_path, exist_ok=True)
    
    plot_true_vs_recovered_params(
        results_df,
        fitted_cfg.PARAM_ORDER,
        results_df_recovered,
        horizon=fitted_cfg.FIT_HORIZON[0],
        path=figure_path,
        is_latex=False,
    )
else:
    print('skipped, no fitted results present')


And a sanity check against real behavior: do the model's simulated draws/outcomes (under the fitted parameters) look like the real subjects' draws/outcomes?

In [ ]:
if HAVE_FIT:
    from src.utils import plot_human_vs_simulated_data, extract_hist_data
    
    human_data = pd.read_pickle(fitted_cfg.HUMAN_DATA_PATH)
    short_draws, long_draws, short_rewards, long_rewards = extract_hist_data(
        human_data[: fitted_cfg.N_SUBJECTS]
    )
    
    all_simulated_data = pd.read_pickle(fitted_cfg.FULL_SIM_DF_PATH)
    outcome_simulated = all_simulated_data["outcome"].values
    num_draws_simulated = np.array([len(ev) for ev in all_simulated_data["ev"].tolist()])
    
    horizon = fitted_cfg.FIT_HORIZON[0]
    human_draws, human_outcome = (long_draws, long_rewards) if horizon == "long" else (short_draws, short_rewards)
    
    plot_human_vs_simulated_data(
        human_outcome,
        human_draws,
        outcome_simulated,
        num_draws_simulated,
        horizon=horizon,
        path=figure_path,
        is_latex=False,
    )
else:
    print('skipped, no fitted results present')


## 7. The four model variants side by side

`POMDPFactory` picks the class from `POMDP_TYPE`. Each variant adds one
mechanism and takes its own parameters on top of the shared ones:

| variant | adds | parameters |
|---|---|---|
| `vanilla` | nothing, the normative agent | `tau`, `xi`, `subjective_cost` |
| `urgency` | temporal regulation | `patience`, `c_max`, `urgency_coefficient`, `urgency_slope` |
| `exaggerate` | transient over-weighting of the newest draw | `exaggeration_factor` |
| `forgetting` | exponential decay of older evidence | `gamma` |

All four also take `belief_bias`, the Beta prior pseudo count on blue.


In [ ]:
import inspect
import numpy as np
from src.pomdp import POMDPFactory

SHARED = dict(
    horizon_condition='short',
    max_cards_per_draw=5,
    is_hazardous=True,
    verbose=False,
    tau=1e-8,          # near deterministic
    xi=0.0,            # no lapses
    subjective_cost=0.0,
    belief_bias=1.0,
)

EXTRA = {
    'vanilla': {},
    'urgency': dict(patience=6.0, c_max=20.0,
                    urgency_coefficient=-10.0, urgency_slope=-2.0),
    'exaggerate': dict(exaggeration_factor=2.5),
}


def build(variant, **overrides):
    """Instantiate and solve one variant, passing only what it accepts."""
    model = POMDPFactory(variant)
    kwargs = {**SHARED, **EXTRA.get(variant, {}), **overrides}
    accepted = set(inspect.signature(type(model).__init__).parameters)
    model.__init__(**{k: v for k, v in kwargs.items() if k in accepted})
    model.value_iteration()
    return model


models = {v: build(v) for v in EXTRA}
for name, m in models.items():
    print(f'{name:12s} {type(m).__name__:22s} action_values {np.asarray(m.action_values).shape}')

# The forgetting variant is the one exception: it reads pre-computed grids of
# discounted evidence, one per gamma, so it needs
#     python3 scripts/generate_gamma_grids.py
# to have been run first. See section 2.3 of PIPELINE.txt.
try:
    models['forgetting'] = build('forgetting', gamma=0.9)
    print(f"{'forgetting':12s} {type(models['forgetting']).__name__:22s} ready")
except (KeyError, FileNotFoundError) as exc:
    print(f'forgetting skipped, the gamma grids are not built yet ({exc!r})')


## 8. Draw the policy with the class method

`plot_best_actions` is the model's own plotting method, so the figures in
the paper and anything you draw here come from the same code. Yellow and
blue mark committing to that colour, green marks waiting.


In [ ]:
import os

figure_dir = os.path.join(project_root, 'figures', 'example_usage')
os.makedirs(figure_dir, exist_ok=True)

models['urgency'].plot_best_actions(
    figsize=(6, 4),
    font_size=11,
    xlabel='draw',
    ylabel='yellow minus blue',
    path=figure_dir,
)


## 9. Replay a specific card sequence

Passing `given_sequence=True` replays exact cards instead of sampling new
ones. Every scoring analysis in the paper works this way, so that the model
and the participant see identical evidence.

A row is `[draw, cumulative_yellow, cumulative_blue, action, outcome]`, with
actions 0 commit yellow, 1 commit blue, 2 wait.


In [ ]:
# five cards per draw, yellow slightly ahead throughout
sequence = [[i + 1, 3 * (i + 1), 2 * (i + 1), 2, 0] for i in range(8)]

result = models['urgency'].simulate_cards_pomdp(
    given_sequence=True, card_sequence=sequence)
print('stopped after', result['num_draws'], 'draws')
print('reward', result['reward'], '  (+2 correct, -2 incorrect, -1 deadline missed)')

# the same sequence always gives the same answer under a deterministic policy
again = models['urgency'].simulate_cards_pomdp(
    given_sequence=True, card_sequence=sequence)
print('repeatable:', again['num_draws'] == result['num_draws'])


## 10. Fit one real subject

The real fits use differential evolution with the budget in
`DE_ALGORITHM_PARAMS`, which takes minutes per subject. `POMDP_DE_PARAMS`
shrinks that so this cell finishes quickly; drop it to reproduce a real fit.

Needs the dataset in `data/TrHu_NHB_light/`, see the README.


In [ ]:
import os
import pandas as pd

# A small budget so this finishes in seconds. Remove it to reproduce a real
# fit, which uses the settings in DE_ALGORITHM_PARAMS and takes minutes.
os.environ['POMDP_DE_PARAMS'] = '{"maxiter": 6, "popsize": 3, "tol": 0.1}'

# Fit the short horizon winner. Its ranges are given explicitly rather than
# taken from src.config, because the default config is a forgetting model and
# would need the gamma grids built first.
SHORT_WINNER_RANGES = {
    'xi': (0, 1),
    'tau': (0, 100),
    'subjective_cost': (-300, 0),
    'patience': (0, 8),
    'belief_bias': (0.01, 5),
}

evidence_path = os.path.join(
    project_root, 'data/TrHu_NHB_light/data_MEG/all_subject_evidence_dicts_short.pkl')

if os.path.exists(evidence_path):
    evidence = pd.read_pickle(evidence_path)
    subject_id = evidence.index[0]
    model = POMDPFactory('urgency')
    best, log_likelihood, *_ = model.fit_subject(
        evidence.loc[subject_id].to_dict(), SHORT_WINNER_RANGES, subject_id, 'de')
    print(f'subject {subject_id}, logL {log_likelihood:.3f}')
    for name, value in best.items():
        print(f'  {name:18s} {value:12.5f}   range {SHORT_WINNER_RANGES[name]}')
else:
    print('dataset not present, see the README for the download steps')


## Where to go next

- `PIPELINE.txt` walks the whole analysis in order, from preprocessing to the
  figure exports.
- `scripts/make_figures.py` produces the manuscript's figures and tables.
- `notebooks/recovery_post_analysis.ipynb` is the full recovery diagnostics that
  section 6 only samples.
- `slurm/` holds one job script per stage, as templates.
- `README.md` covers install, the data, and the fitting workflow.